# APKGC — Ablation Bảng 4 (bản Colab)

Kiểm chứng độc lập hai đóng góp của bài báo trên MKG-W.

| Cấu hình | Noise | Penalty |
|---|---|---|
| `abl_full` | bật | bật |
| `abl_no_noise` | **tắt** | bật |
| `abl_no_penalty` | bật | **tắt** |
| `abl_neither` | tắt | tắt |

Điều kiện giống hệt "đợt 2" đã chạy (2000 epoch, đánh giá mỗi 250, seed 42) để so
được với mốc tự đo **MRR 0,3605 ± 0,0009**.

**Colab hay ngắt giữa chừng** nên notebook này lưu kết quả vào Google Drive và bỏ
qua thí nghiệm đã xong — chạy lại ô cuối là tiếp tục được.

In [ ]:
# Kết quả lưu vào Drive để sống sót qua lần ngắt kết nối.
# Colab free thường ngắt sau vài giờ, mà một lần chạy ~2,6 giờ.
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = "/content/drive/MyDrive/apkgc_ablation"
os.makedirs(OUT, exist_ok=True)
print("kết quả sẽ nằm ở:", OUT)
print("đã có:", sorted(f for f in os.listdir(OUT) if f.endswith(".json")))

In [ ]:
import torch, subprocess, sys
print("torch", torch.__version__)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    archs = torch.cuda.get_arch_list()
    print(f"GPU {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  hỗ trợ: {archs}")
    NEED_DOWNGRADE = f"sm_{cap[0]}{cap[1]}" not in archs
else:
    raise SystemExit("KHÔNG CÓ GPU — Runtime > Change runtime type > GPU")
print("cần hạ torch:", NEED_DOWNGRADE)

In [ ]:
# Colab thường cấp T4 (sm_75) — torch bản mới hỗ trợ sẵn, không cần hạ.
# Giữ nhánh này phòng khi được cấp GPU cũ hơn.
if NEED_DOWNGRADE:
    !pip install -q --force-reinstall torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -2
    r = subprocess.run([sys.executable,"-c",
        "import torch;a=torch.randn(8,8,device='cuda');print('matmul OK',float((a@a).sum()))"],
        capture_output=True, text=True)
    print(r.stdout or r.stderr[-300:])
else:
    print("Không cần thay đổi torch.")

In [ ]:
%cd /content
!rm -rf APKGC _openke
!git clone -q https://github.com/HubuKG/APKGC.git
!pip install -q transformers easydict unidecode tensorboard scikit-learn scipy 2>&1 | tail -1
import os
print("clone xong:", os.path.exists("/content/APKGC/run_KGC_train.py"))

In [ ]:
import os, glob, subprocess, sys
A = "/content/APKGC"

# 1. Bổ sung mã nguồn C++ còn thiếu. Repo chỉ commit Base.so biên dịch sẵn;
#    thư mục mmkgc/base/ chứa Base.cpp không được commit nên không biên dịch lại
#    được. mmkgc/ là bản sửa đổi của OpenKE (THUNLP) nên lấy base/ từ đó.
if not os.path.exists(f"{A}/mmkgc/base/Base.cpp"):
    subprocess.run(["git","clone","-q","--depth","1",
                    "https://github.com/thunlp/OpenKE.git","/content/_openke"], check=True)
    subprocess.run(["cp","-r","/content/_openke/openke/base", f"{A}/mmkgc/base"], check=True)
    print("đã bổ sung mmkgc/base/ từ OpenKE")

# 2. Module chọn thiết bị
open(f"{A}/mmkgc/device.py","w").write("\n".join([
    "import os", "import torch",
    '_f = os.environ.get("MMKGC_DEVICE", "").strip().lower()',
    "if _f:", "    DEVICE = torch.device(_f)",
    "elif torch.cuda.is_available():", '    DEVICE = torch.device("cuda")',
    "else:", '    DEVICE = torch.device("cpu")',
    "def device_name():", "    return str(DEVICE)", ""]))

# 3. Thay .cuda() cứng
total = 0
for p in glob.glob(f"{A}/**/*.py", recursive=True):
    if p.endswith("device.py"): continue
    s = open(p, encoding="utf-8").read()
    if ".cuda()" not in s: continue
    total += s.count(".cuda()")
    rel = os.path.relpath(p, A)
    if rel.startswith("mmkgc/"):
        depth = os.path.relpath(p, f"{A}/mmkgc").count(os.sep)
        imp = "from " + "."*(depth+1) + "device import DEVICE"
    else:
        imp = "from mmkgc.device import DEVICE"
    if "device import DEVICE" not in s:
        lines = s.split("\n")
        last = max((i for i,l in enumerate(lines) if l.startswith(("import ","from "))), default=0)
        lines.insert(last+1, imp); s = "\n".join(lines)
    open(p,"w",encoding="utf-8").write(s.replace(".cuda()", ".to(DEVICE)"))
print(f"đã thay {total} chỗ .cuda()")

# 4. Sửa các lỗi khiến không chạy được
p = f"{A}/mmkgc/config/Trainer.py"; s = open(p, encoding="utf-8").read()
s = s.replace("data_loader=self.test_dataloader, use_gpu=True)",
              "data_loader=self.test_dataloader, use_gpu=(DEVICE.type != 'cpu'))")
if "self.best_model_wts = None" not in s:
    s = s.replace("        self.Loss_log = Loss_log(early_stop=args.early_stop)",
                  "        self.Loss_log = Loss_log(early_stop=args.early_stop)\n"
                  "        self.best_model_wts = None")
open(p,"w",encoding="utf-8").write(s)

p = f"{A}/args.py"; s = open(p, encoding="utf-8").read()
if "save_steps" not in s:
    s = s.replace("    arg.add_argument('-num_proj', type=int, default=1)",
                  "    arg.add_argument('-num_proj', type=int, default=1)\n"
                  "    arg.add_argument('-save_steps', type=int, default=100)")
open(p,"w",encoding="utf-8").write(s)

p = f"{A}/run_KGC_train.py"; s = open(p, encoding="utf-8").read()
s = s.replace("    torch.cuda.manual_seed_all(args.seed)",
              "    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)")
s = s.replace("    data_root = osp.abspath(osp.join(this_dir, '..', '..', 'data', ''))",
              "    data_root = osp.abspath(osp.join(this_dir, 'output'))")
s = s.replace("        save_steps=100,", "        save_steps=args.save_steps,")
s = s.replace("use_gpu=True", "use_gpu=(DEVICE.type != 'cpu')")
if "from mmkgc.device import DEVICE" not in s:
    s = s.replace("from args import get_args",
                  "from args import get_args\nfrom mmkgc.device import DEVICE, device_name")
OLD = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if not osp.exists(save_dir):",
    '        torch.save(trainer.best_model_wts, f"{args.save}-MRR{trainer.Loss_log.get_acc()}")',
    "", "    kge_score.load_checkpoint(save_dir)"])
NEW = "\n".join([
    '    save_dir = f"{args.save}-MRR{trainer.Loss_log.get_acc()}"',
    "    if trainer.best_model_wts is not None:",
    "        if not osp.exists(save_dir):",
    "            torch.save(trainer.best_model_wts, save_dir)",
    "        kge_score.load_checkpoint(save_dir)"])
if OLD in s: s = s.replace(OLD, NEW)
open(p,"w",encoding="utf-8").write(s)
print("đã sửa Trainer/args/entrypoint")

In [ ]:
%cd /content/APKGC/mmkgc
!bash make.sh 2>&1 | tail -3
%cd /content
import os
print("Base.so:", os.path.exists("/content/APKGC/mmkgc/release/Base.so"))

In [ ]:
# Bộ nhúng MKG-W của tác giả
!pip install -q gdown 2>&1 | tail -1
%cd /content/APKGC
!gdown --folder 1wGvlzt0i6RYcMniwooFcLKVEs2NBMcpB -O ./emb_dl -q
import shutil, pathlib
pathlib.Path('embeddings').mkdir(exist_ok=True)
for f in pathlib.Path('emb_dl').rglob('*'):
    if f.suffix in ('.pth','.json'):
        dst = pathlib.Path('embeddings')/f.name
        if not dst.exists(): shutil.move(str(f), str(dst))
%cd /content
!ls -la /content/APKGC/embeddings/

In [ ]:
# ABLATION — vô hiệu hoá attention penalty.
# Không có cờ dòng lệnh nào tắt được: APKGC.py luôn tính penalty rồi so với
# self.penalty_threshold = 10 (cứng hoá). Cách ablate sạch nhất là nâng ngưỡng
# lên rất cao => exceed_threshold_mask luôn False => KHÔNG phân phối lại trọng
# số. Phần tính penalty vẫn chạy nên chi phí tính toán không đổi, chỉ tác dụng
# bị vô hiệu — đúng nghĩa ablation.
import re
P = f"{A}/mmkgc/module/model/APKGC.py"
assert "self.penalty_threshold = 10" in open(P).read(), "không tìm thấy dòng đặt ngưỡng"

def set_penalty(enabled: bool):
    s = open(P, encoding="utf-8").read()
    s = re.sub(r"self\.penalty_threshold = [0-9.e+]+",
               f"self.penalty_threshold = {'10' if enabled else '1e30'}", s)
    open(P, "w", encoding="utf-8").write(s)
    now = re.search(r"self\.penalty_threshold = ([0-9.e+]+)", open(P).read()).group(1)
    print(f"  penalty {'BẬT' if enabled else 'TẮT'} -> ngưỡng = {now}")

set_penalty(True); set_penalty(False); set_penalty(True)   # thử cả hai chiều

In [ ]:
import subprocess, sys, os, re, json, time
LOGGER_LINE = re.compile(r"mrr:\s*([\d.]+).*?mr:\s*([\d.]+).*?hit10:\s*([\d.]+)"
                         r".*?hit3:\s*([\d.]+).*?hit1:\s*([\d.]+)")
SHELL_VAR = re.compile(r"^\s*([A-Z_][A-Z0-9_]*)=(\S+)\s*$", re.MULTILINE)

def read_author_config():
    """Đọc siêu tham số từ chính script tác giả, không tự chế số."""
    cfg = dict(SHELL_VAR.findall(open(f"{A}/scripts/run_base.sh").read()))
    for k in ["EMB_DIM","NUM_BATCH","MARGIN","LR","NEG_NUM","EPOCH"]:
        assert k in cfg, f"thiếu {k}"
    return cfg

def run_exp(tag, cfg, seed=42, epoch=2000, save_steps=250, add_noise=1):
    # Bỏ qua nếu đã chạy xong ở phiên trước — Colab hay ngắt giữa chừng.
    done = f"{OUT}/{tag}.json"
    if os.path.exists(done):
        d = json.load(open(done))
        print(f"  [{tag}] đã có: MRR {d['mrr']:.4f} — bỏ qua"); return d
    os.makedirs(f"{A}/output/mmkgc/checkpoint", exist_ok=True)
    cmd = [sys.executable, "-u", "run_KGC_train.py",
           "-dataset=MKG-W", "-num_proj=1", "-use_intermediate=0",
           "-joint_way=Mformer_hd_mean",
           f"-batch_size={cfg['NUM_BATCH']}", f"-margin={cfg['MARGIN']}",
           f"-epoch={epoch}", f"-dim={cfg['EMB_DIM']}",
           f"-neg_num={cfg['NEG_NUM']}", f"-learning_rate={cfg['LR']}",
           f"-noise_update={cfg.get('NU','epoch')}",
           "-noise_ratio=0.2", "-mask_ratio=0.7",
           f"-use_pool={cfg.get('POOL',1)}", f"-add_noise={add_noise}",
           "-num_hidden_layers=1", "-num_attention_heads=2",
           f"-seed={seed}", f"-save={tag}",
           f"-save_steps={save_steps}", f"-exp_id={tag}"]
    print(f"  [{tag}] bắt đầu ...", flush=True)
    t0 = time.time(); evals = []
    with open(f"{OUT}/{tag}.log","w",buffering=1) as lf:
        proc = subprocess.Popen(cmd, cwd=A, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            lf.write(line)
            m = LOGGER_LINE.search(line)
            if m:
                ev = dict(mrr=float(m.group(1)), mr=float(m.group(2)),
                          hit10=float(m.group(3)), hit3=float(m.group(4)),
                          hit1=float(m.group(5)))
                evals.append(ev)
                print(f"    eval {len(evals):>2}: MRR {ev['mrr']:.4f}", flush=True)
                # ghi ngay vào Drive để mất kết nối vẫn còn
                json.dump(dict(tag=tag, evals=evals), open(f"{OUT}/partial_{tag}.json","w"))
        proc.wait()
    if not evals:
        print(f"  [{tag}] KHÔNG CÓ KẾT QUẢ — xem {OUT}/{tag}.log"); return None
    res = dict(evals[-1]); res.update(tag=tag, seed=seed, add_noise=add_noise,
               n_evals=len(evals), seconds=round(time.time()-t0,1),
               epoch=epoch, save_steps=save_steps,
               penalty_threshold=re.search(r"self\.penalty_threshold = ([0-9.e+]+)",
                                            open(P).read()).group(1))
    json.dump(res, open(done,"w"), indent=2)
    print(f"  [{tag}] xong: MRR {res['mrr']:.4f} sau {res['seconds']/60:.0f} phút")
    return res

cfg = read_author_config()
print("cấu hình đọc từ scripts/run_base.sh:", cfg)

## Chạy

Bốn thí nghiệm, mỗi cái ~2,6 giờ. Colab free hiếm khi giữ được 10 giờ liền, nên
**chạy ô dưới nhiều lần** — mỗi lần nó bỏ qua phần đã xong và làm tiếp phần còn
thiếu. Kết quả nằm trên Drive nên không mất.

In [ ]:
PLAN = [("abl_full",       1, True),
        ("abl_no_noise",   0, True),
        ("abl_no_penalty", 1, False),
        ("abl_neither",    0, False)]

for tag, noise, penalty_on in PLAN:
    set_penalty(penalty_on)
    run_exp(tag, cfg, seed=42, add_noise=noise)

In [ ]:
import glob, json
PUB_GAIN = {"noise": 0.016, "penalty": 0.014}   # bài báo, Bảng 4
SEED_STD = 0.0009                                # tự đo, đợt 2 n=3
BASE_OWN = 0.3605                                # tự đo, đợt 2 trung bình

r = {}
print("=== KẾT QUẢ ABLATION ===")
for f in sorted(glob.glob(f"{OUT}/abl_*.json")):
    if "partial" in f: continue
    d = json.load(open(f)); r[d["tag"]] = d
    print(f"  {d['tag']:16} MRR {d['mrr']:.4f}  H@1 {d['hit1']:.4f}  "
          f"H@3 {d['hit3']:.4f}  H@10 {d['hit10']:.4f}  "
          f"(noise={d['add_noise']}, ngưỡng={d['penalty_threshold']})")

if "abl_full" in r and "abl_no_noise" in r:
    g = r["abl_full"]["mrr"] - r["abl_no_noise"]["mrr"]
    print(f"\nĐóng góp noise sampling  : {g:+.4f}  "
          f"(bài báo {PUB_GAIN['noise']:+.3f}, gấp {abs(g)/SEED_STD:.1f}× nhiễu seed)")
if "abl_full" in r and "abl_no_penalty" in r:
    g = r["abl_full"]["mrr"] - r["abl_no_penalty"]["mrr"]
    print(f"Đóng góp attention penalty: {g:+.4f}  "
          f"(bài báo {PUB_GAIN['penalty']:+.3f}, gấp {abs(g)/SEED_STD:.1f}× nhiễu seed)")
print(f"\nMốc tự đo (đợt 2, n=3): MRR {BASE_OWN:.4f} ± {SEED_STD:.4f}")
print(f"Còn thiếu: {[t for t,_,_ in PLAN if t not in r]}")